In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import joblib



In [2]:
import pandas as pd
import os

# Define relative path
csv_path = os.path.join("..", "data", "processed", "multi_ticker_options_clean.csv")

# Load CSV
df = pd.read_csv(csv_path)

# Confirm load
print("✅ Data loaded. Shape:", df.shape)
df.head()


✅ Data loaded. Shape: (1869, 20)


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,ticker,expiry,days_to_expiry,spot_price,moneyness,date_collected
0,SPY250630C00345000,2025-06-03 19:58:00+00:00,345.0,252.03,268.41,270.54,0.000000,0.00000,12.0,0,3.583985,True,REGULAR,USD,SPY,2025-06-30,0,614.909973,0.561058,2025-06-29 12:49:52.353214
1,SPY250630C00350000,2025-06-27 19:45:05+00:00,350.0,263.21,263.41,265.55,12.999985,5.19563,5.0,0,3.510743,True,REGULAR,USD,SPY,2025-06-30,0,614.909973,0.569189,2025-06-29 12:49:52.353214
2,SPY250630C00375000,2025-06-26 15:09:29+00:00,375.0,235.73,238.42,240.55,0.000000,0.00000,1.0,0,3.123049,True,REGULAR,USD,SPY,2025-06-30,0,614.909973,0.609845,2025-06-29 12:49:52.353214
3,SPY250630C00380000,2025-05-23 15:41:23+00:00,380.0,200.19,214.11,216.10,0.000000,0.00000,20.0,0,0.000010,True,REGULAR,USD,SPY,2025-06-30,0,614.909973,0.617977,2025-06-29 12:49:52.353214
4,SPY250630C00385000,2025-06-24 19:51:02+00:00,385.0,220.98,228.42,230.55,0.000000,0.00000,1.0,1,2.974612,True,REGULAR,USD,SPY,2025-06-30,0,614.909973,0.626108,2025-06-29 12:49:52.353214


In [4]:
from scipy.stats import norm
import numpy as np

def black_scholes_price(S, K, T, r, sigma, option_type="call"):
    # Avoid divide-by-zero errors
    if sigma <= 0 or T <= 0:
        return max(S - K, 0) if option_type == "call" else max(K - S, 0)

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


In [5]:
r = 0.05  # risk-free rate

df["bs_price"] = df.apply(lambda row: black_scholes_price(
    row["spot_price"],
    row["strike"],
    row["days_to_expiry"] / 365,
    r,
    row["impliedVolatility"]
), axis=1)

print("✅ Black-Scholes prices added.")


✅ Black-Scholes prices added.


In [7]:
def monte_carlo_price(S, K, T, r, sigma, sims=10000, option_type="call"):
    if sigma <= 0 or T <= 0:
        return max(S - K, 0) if option_type == "call" else max(K - S, 0)

    np.random.seed(0)  # For reproducibility
    Z = np.random.standard_normal(sims)
    ST = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    if option_type == "call":
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)

    return np.exp(-r * T) * np.mean(payoffs)



In [8]:
df["mc_price"] = df.apply(lambda row: monte_carlo_price(
    row["spot_price"],
    row["strike"],
    row["days_to_expiry"] / 365,
    r,
    row["impliedVolatility"]
), axis=1)

print("✅ Monte Carlo prices added.")


✅ Monte Carlo prices added.


In [10]:
df["bs_diff"] = df["lastPrice"] - df["bs_price"]
df["mc_diff"] = df["lastPrice"] - df["mc_price"]
df["bs_vs_mc"] = df["bs_price"] - df["mc_price"]


In [13]:
# Recreate the binary label
df["label"] = (df["lastPrice"] > df["bid"]).astype(int)


In [15]:
features = df.select_dtypes(include=[np.number]).drop(columns=["label"], errors="ignore")
target = df["label"]



In [16]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Scale numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# Fix imbalance with SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, target)



In [17]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier

X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

model = LGBMClassifier(random_state=42)
model.fit(X_train, y_train)


[LightGBM] [Info] Number of positive: 919, number of negative: 932
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002064 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3882
[LightGBM] [Info] Number of data points in the train set: 1851, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.496488 -> initscore=-0.014047
[LightGBM] [Info] Start training from score -0.014047


LGBMClassifier(random_state=42)

In [19]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 👇 Fix the warning
X_test = pd.DataFrame(X_test, columns=features.columns)

# 🔮 Predict and evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# 🧪 Evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("🔍 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

print("\n🔥 ROC AUC Score:")
print("ROC AUC:", roc_auc_score(y_test, y_proba))




🔍 Confusion Matrix:
[[206  19]
 [ 15 223]]

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.92      0.92       225
           1       0.92      0.94      0.93       238

    accuracy                           0.93       463
   macro avg       0.93      0.93      0.93       463
weighted avg       0.93      0.93      0.93       463


🔥 ROC AUC Score:
ROC AUC: 0.9796825396825397


In [20]:
joblib.dump(model, "models/final_lightgbm_with_quant_features.pkl")


['models/final_lightgbm_with_quant_features.pkl']